<a href="https://colab.research.google.com/github/vsolanki76771215/Student_MLE_MiniProject_EDA/blob/main/ExperimentWithVariousModels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# STEP 1: Import Required Libraries
# ============================================================

import os
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    VotingClassifier,
    StackingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

In [ ]:
# ============================================================
# STEP 2: Load Dataset from Colab sample_data Folder
# ============================================================

# In this project, the processed machine learning dataset has already
# been uploaded to the Colab environment and stored in the sample_data
# directory. We will read the CSV directly from that location.

# Specify the dataset location inside sample_data.
# Replace the filename if your uploaded file has a different name.

DATA_PATH = "/content/sample_data/all_patches_features_labels_s2_ndvi.csv"

# Verify that the file exists before loading.

if os.path.exists(DATA_PATH):
    print("Dataset found.")
else:
    print("Dataset not found. Check filename and location.")

df = pd.read_csv(DATA_PATH)

# Display basic information about the dataset.

print("Dataset Shape:", df.shape)

# Preview first few records.

df.head()


Dataset found.
Dataset Shape: (91454, 8)


,aoi,patch_file,ndvi_2018_mean,ndvi_2022_mean,ndvi_diff,loss_fraction,loss_binary,treecover_mean
0,la_pampa,patch_000000.npz,NaN,0.734583,NaN,0.000000,0,99.921875
1,la_pampa,patch_000001.npz,NaN,0.330006,NaN,0.302734,1,99.710938
2,la_pampa,patch_000002.npz,NaN,0.951192,NaN,0.314453,1,97.947266
3,la_pampa,patch_000003.npz,NaN,0.955078,NaN,0.034180,0,99.526367
4,la_pampa,patch_000004.npz,NaN,0.986328,NaN,0.000000,0,99.088867


In [ ]:
# ============================================================
# STEP 3: Create Output Directory for Model Results
# ============================================================

# All experiment outputs, tuning results, trained models,
# and evaluation summaries will be saved in this folder.

OUTPUT_DIR = "/content/model_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output Directory:", OUTPUT_DIR)

Output Directory: /content/model_results


In [ ]:
# ============================================================
# STEP 4: Inspect Dataset
# ============================================================

print("Columns:")
print(df.columns)

print("\nMissing values:")
print(df.isna().sum())

print("\nClass balance:")
print(df["loss_binary"].value_counts())
print(df["loss_binary"].value_counts(normalize=True))

Columns:
Index(['aoi', 'patch_file', 'ndvi_2018_mean', 'ndvi_2022_mean', 'ndvi_diff',
       'loss_fraction', 'loss_binary', 'treecover_mean'],
      dtype='object')

Missing values:
aoi                   0
patch_file            0
ndvi_2018_mean    23173
ndvi_2022_mean        0
ndvi_diff         23173
loss_fraction         0
loss_binary           0
treecover_mean        0
dtype: int64

Class balance:
loss_binary
0    79493
1    11961
Name: count, dtype: int64
loss_binary
0    0.869213
1    0.130787
Name: proportion, dtype: float64


In [ ]:
# ============================================================
# STEP 5: Define Features and Target
# ============================================================

# Features are NDVI-based satellite measurements.
# Target is binary deforestation label:
# 0 = no major forest loss
# 1 = forest loss detected

feature_cols = [
    "ndvi_2018_mean",
    "ndvi_2022_mean",
    "ndvi_diff"
]

target_col = "loss_binary"

df_model = df.dropna(subset=feature_cols + [target_col])

X = df_model[feature_cols]
y = df_model[target_col].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (68281, 3)
y shape: (68281,)


In [ ]:
# ============================================================
# STEP 6: Train/Test Split
# ============================================================

# The train/test split gives us a final holdout test set.
# Cross-validation will be applied only on the training set.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (54624, 3)
Testing set: (13657, 3)


In [ ]:
# ============================================================
# STEP 7: Define Performance Metrics
# ============================================================

# F1 is the main metric because deforestation detection may have class imbalance.
# Accuracy alone can be misleading when most patches are non-deforestation.

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

In [ ]:
# ============================================================
# STEP 8: Define Multiple Models
# ============================================================

# Some models need scaling, such as Logistic Regression, SVC, KNN, and MLP.
# Tree-based models do not require scaling.

models = {
    "Dummy Baseline": DummyClassifier(strategy="most_frequent"),

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Decision Tree Gini": DecisionTreeClassifier(
        criterion="gini",
        class_weight="balanced",
        random_state=42
    ),

    "Decision Tree Entropy": DecisionTreeClassifier(
        criterion="entropy",
        class_weight="balanced",
        random_state=42
    ),

    "Decision Tree Log Loss": DecisionTreeClassifier(
        criterion="log_loss",
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        loss="log_loss",
        random_state=42
    ),

    "AdaBoost": AdaBoostClassifier(
        random_state=42
    ),

    "SVC": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            class_weight="balanced",
            probability=True,
            random_state=42
        ))
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),

    "Gaussian NB": GaussianNB(),

    "MLP Neural Network": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            max_iter=300,
            random_state=42
        ))
    ])
}

In [ ]:
# ============================================================
# STEP 9: Automated Model Testing with K-Fold Cross Validation
# ============================================================

results = []

for model_name, model in models.items():
    print(f"Training and evaluating: {model_name}")

    start_time = time.time()

    cv_scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )

    training_time = time.time() - start_time

    result = {
        "model": model_name,
        "training_time_seconds": training_time,

        "train_accuracy": cv_scores["train_accuracy"].mean(),
        "test_accuracy": cv_scores["test_accuracy"].mean(),

        "train_precision": cv_scores["train_precision"].mean(),
        "test_precision": cv_scores["test_precision"].mean(),

        "train_recall": cv_scores["train_recall"].mean(),
        "test_recall": cv_scores["test_recall"].mean(),

        "train_f1": cv_scores["train_f1"].mean(),
        "test_f1": cv_scores["test_f1"].mean(),

        "train_roc_auc": cv_scores["train_roc_auc"].mean(),
        "test_roc_auc": cv_scores["test_roc_auc"].mean(),

        "train_pr_auc": cv_scores["train_pr_auc"].mean(),
        "test_pr_auc": cv_scores["test_pr_auc"].mean(),

        "overfit_gap_f1": cv_scores["train_f1"].mean() - cv_scores["test_f1"].mean()
    }

    results.append(result)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="test_f1", ascending=False)

results_df

Training and evaluating: Dummy Baseline
Training and evaluating: Logistic Regression
Training and evaluating: Decision Tree Gini
Training and evaluating: Decision Tree Entropy
Training and evaluating: Decision Tree Log Loss
Training and evaluating: Random Forest
Training and evaluating: Extra Trees


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Training and evaluating: Gradient Boosting
Training and evaluating: AdaBoost
Training and evaluating: SVC


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Training and evaluating: KNN
Training and evaluating: Gaussian NB
Training and evaluating: MLP Neural Network


,model,training_time_seconds,train_accuracy,test_accuracy,train_precision,test_precision,train_recall,test_recall,train_f1,test_f1,train_roc_auc,test_roc_auc,train_pr_auc,test_pr_auc,overfit_gap_f1
9,SVC,3404.937570,0.788303,0.787291,0.329844,0.327632,0.567530,0.563827,0.417204,0.414416,0.734022,0.731835,0.331535,0.329326,0.002789
1,Logistic Regression,0.648104,0.798312,0.797873,0.319248,0.318294,0.450946,0.450019,0.373836,0.372836,0.690355,0.690284,0.245174,0.245607,0.000999
4,Decision Tree Log Loss,3.017956,0.987368,0.820445,0.940058,0.325990,0.967092,0.322638,0.953373,0.324238,0.998781,0.607479,0.991932,0.205768,0.629135
3,Decision Tree Entropy,2.512100,0.987368,0.820445,0.940058,0.325990,0.967092,0.322638,0.953373,0.324238,0.998781,0.607479,0.991932,0.205768,0.629135
2,Decision Tree Gini,2.015727,0.987368,0.818834,0.940058,0.321672,0.967092,0.321537,0.953373,0.321497,0.998781,0.606047,0.991933,0.203997,0.631876
6,Extra Trees,100.115712,0.987368,0.837727,0.940058,0.363033,0.967092,0.284244,0.953373,0.318767,0.998781,0.722785,0.991924,0.293949,0.634606
5,Random Forest,153.489484,0.988970,0.846203,0.954567,0.383754,0.963252,0.249417,0.958884,0.302226,0.997985,0.737007,0.984621,0.321598,0.656658
10,KNN,4.520843,0.891366,0.858542,0.687792,0.440736,0.341354,0.220487,0.456224,0.293821,0.890337,0.701297,0.516800,0.274439,0.162403
12,MLP Neural Network,66.309674,0.870149,0.869343,0.557368,0.545677,0.135061,0.130125,0.217175,0.209860,0.757193,0.754590,0.377228,0.370772,0.007315
7,Gradient Boosting,36.537313,0.871787,0.869416,0.622537,0.569586,0.100679,0.090634,0.173247,0.156351,0.783136,0.770306,0.400073,0.371553,0.016896


In [ ]:
# ============================================================
# STEP 10: Save Experiment Results
# ============================================================

results_path = os.path.join(OUTPUT_DIR, "experiment_results.csv")
results_df.to_csv(results_path, index=False)

print("Saved experiment results to:")
print(results_path)

Saved experiment results to:
/content/model_results/experiment_results.csv


In [ ]:
# ============================================================
# STEP 11: Hyperparameter Tuning
# ============================================================

# We tune Random Forest because it is usually strong for tabular data.
# We test multiple hyperparameters and different split criteria/loss-style options.

rf_model = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "criterion": ["gini", "entropy", "log_loss"]
}

grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print("Best cross-validation F1:")
print(grid_search.best_score_)

Fitting 5 folds for each of 324 candidates, totalling 1620 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
# ============================================================
# STEP 11: Faster Hyperparameter Tuning with RandomizedSearchCV
# ============================================================

# The previous GridSearchCV tested 324 parameter combinations.
# With 5-fold cross-validation, that became:
# 324 x 5 = 1620 model fits.
#
# That is too slow for Colab and can run for many hours.
#
# RandomizedSearchCV tests only a selected number of random combinations.
# This is much faster and still satisfies the project requirement for
# hyperparameter tuning across multiple parameters.

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import os
import joblib
import time

rf_model = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=1
)

param_dist = {
    "n_estimators": [50, 100, 150],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "criterion": ["gini", "entropy"]
}

start_time = time.time()

random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_dist,
    n_iter=15,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=1,
    verbose=2,
    return_train_score=True
)

random_search.fit(X_train, y_train)

tuning_time = time.time() - start_time

print("Best parameters:")
print(random_search.best_params_)

print("Best cross-validation F1:")
print(random_search.best_score_)

print("Tuning time in seconds:")
print(tuning_time)

Fitting 3 folds for each of 15 candidates, totalling 45 fits
[CV] END criterion=entropy, max_depth=None, min_samples_leaf=2, min_samples_split=5, n_estimators=50; total time=   6.0s
[CV] END criterion=entropy, max_depth=None, min_samples_leaf=2, min_samples_split=5, n_estimators=50; total time=   5.0s
[CV] END criterion=entropy, max_depth=None, min_samples_leaf=2, min_samples_split=5, n_estimators=50; total time=   6.0s
[CV] END criterion=entropy, max_depth=None, min_samples_leaf=4, min_samples_split=10, n_estimators=50; total time=   4.7s
[CV] END criterion=entropy, max_depth=None, min_samples_leaf=4, min_samples_split=10, n_estimators=50; total time=   4.8s
[CV] END criterion=entropy, max_depth=None, min_samples_leaf=4, min_samples_split=10, n_estimators=50; total time=   5.2s
[CV] END criterion=entropy, max_depth=10, min_samples_leaf=1, min_samples_split=5, n_estimators=50; total time=   3.1s
[CV] END criterion=entropy, max_depth=10, min_samples_leaf=1, min_samples_split=5, n_estima

In [ ]:
# ============================================================
# STEP 12: Save Best Tuned Random Forest Model
# ============================================================

best_rf_model = random_search.best_estimator_

best_rf_path = os.path.join(OUTPUT_DIR, "best_random_forest_model.pkl")
joblib.dump(best_rf_model, best_rf_path)

print("Saved best Random Forest model to:")
print(best_rf_path)

Saved best Random Forest model to:
/content/model_results/best_random_forest_model.pkl


In [ ]:
# ============================================================
# STEP 13: Save Randomized Search Tuning Results
# ============================================================

tuning_results_df = pd.DataFrame(random_search.cv_results_)

tuning_results_path = os.path.join(
    OUTPUT_DIR,
    "random_forest_randomized_search_results.csv"
)

tuning_results_df.to_csv(tuning_results_path, index=False)

print("Saved tuning results to:")
print(tuning_results_path)

Saved tuning results to:
/content/model_results/random_forest_randomized_search_results.csv


In [ ]:
# ============================================================
# STEP 14: Build Colab-Friendly Ensemble Models
# ============================================================

# Stacking is computationally expensive because it trains multiple
# models inside each cross-validation fold.
#
# To keep the notebook stable in Colab, this version uses:
# 1. A lightweight soft voting ensemble
# 2. Fewer trees
# 3. n_jobs=1
# 4. No stacking model

rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=1
)

et = ExtraTreesClassifier(
    n_estimators=50,
    max_depth=10,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=1
)

gb = GradientBoostingClassifier(
    loss="log_loss",
    n_estimators=50,
    max_depth=3,
    random_state=42
)

lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

soft_voting_model = VotingClassifier(
    estimators=[
        ("random_forest", rf),
        ("extra_trees", et),
        ("gradient_boosting", gb),
        ("logistic_regression", lr)
    ],
    voting="soft",
    n_jobs=1
)

ensemble_models = {
    "Soft Voting Ensemble": soft_voting_model
}

In [ ]:
# ============================================================
# STEP 15: Evaluate Colab-Friendly Ensemble Model
# ============================================================

ensemble_results = []

for model_name, model in ensemble_models.items():
    print(f"Evaluating ensemble: {model_name}")

    start_time = time.time()

    cv_scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=3,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1
    )

    training_time = time.time() - start_time

    result = {
        "model": model_name,
        "training_time_seconds": training_time,
        "train_f1": cv_scores["train_f1"].mean(),
        "test_f1": cv_scores["test_f1"].mean(),
        "test_precision": cv_scores["test_precision"].mean(),
        "test_recall": cv_scores["test_recall"].mean(),
        "test_accuracy": cv_scores["test_accuracy"].mean(),
        "test_roc_auc": cv_scores["test_roc_auc"].mean(),
        "test_pr_auc": cv_scores["test_pr_auc"].mean(),
        "overfit_gap_f1": cv_scores["train_f1"].mean() - cv_scores["test_f1"].mean()
    }

    ensemble_results.append(result)

ensemble_results_df = pd.DataFrame(ensemble_results)
ensemble_results_df = ensemble_results_df.sort_values(by="test_f1", ascending=False)

ensemble_results_df

Evaluating ensemble: Soft Voting Ensemble


,model,training_time_seconds,train_f1,test_f1,test_precision,test_recall,test_accuracy,test_roc_auc,test_pr_auc,overfit_gap_f1
0,Soft Voting Ensemble,22.895769,0.456945,0.425248,0.384458,0.475799,0.828317,0.775987,0.364421,0.031698


In [ ]:
# ============================================================
# STEP 16: Save Ensemble Results
# ============================================================

ensemble_results_path = os.path.join(OUTPUT_DIR, "ensemble_results.csv")
ensemble_results_df.to_csv(ensemble_results_path, index=False)

print("Saved ensemble results to:")
print(ensemble_results_path)

Saved ensemble results to:
/content/model_results/ensemble_results.csv


In [ ]:
# ============================================================
# STEP 17: Final Model Selection
# ============================================================

# Combine regular model results and ensemble results.
# Select the model with the highest cross-validation F1 score.

regular_results = results_df[[
    "model",
    "training_time_seconds",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_roc_auc",
    "test_pr_auc",
    "overfit_gap_f1"
]]

all_results = pd.concat(
    [regular_results, ensemble_results_df],
    ignore_index=True
)

all_results = all_results.sort_values(by="test_f1", ascending=False)

all_results

,model,training_time_seconds,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc,overfit_gap_f1,train_f1
13,Soft Voting Ensemble,22.895769,0.828317,0.384458,0.475799,0.425248,0.775987,0.364421,0.031698,0.456945
0,SVC,3404.937570,0.787291,0.327632,0.563827,0.414416,0.731835,0.329326,0.002789,NaN
1,Logistic Regression,0.648104,0.797873,0.318294,0.450019,0.372836,0.690284,0.245607,0.000999,NaN
2,Decision Tree Log Loss,3.017956,0.820445,0.325990,0.322638,0.324238,0.607479,0.205768,0.629135,NaN
3,Decision Tree Entropy,2.512100,0.820445,0.325990,0.322638,0.324238,0.607479,0.205768,0.629135,NaN
4,Decision Tree Gini,2.015727,0.818834,0.321672,0.321537,0.321497,0.606047,0.203997,0.631876,NaN
5,Extra Trees,100.115712,0.837727,0.363033,0.284244,0.318767,0.722785,0.293949,0.634606,NaN
6,Random Forest,153.489484,0.846203,0.383754,0.249417,0.302226,0.737007,0.321598,0.656658,NaN
7,KNN,4.520843,0.858542,0.440736,0.220487,0.293821,0.701297,0.274439,0.162403,NaN
8,MLP Neural Network,66.309674,0.869343,0.545677,0.130125,0.209860,0.754590,0.370772,0.007315,NaN


In [ ]:
# ============================================================
# STEP 18: Train Final Best Model on Full Training Data
# ============================================================

best_model_name = all_results.iloc[0]["model"]

print("Best model based on cross-validation F1:")
print(best_model_name)

# Choose actual model object.
# If Random Forest tuning performed better, use best_rf_model.
# Otherwise, choose from the model dictionary or ensemble dictionary.

if best_model_name in models:
    final_model = models[best_model_name]
elif best_model_name in ensemble_models:
    final_model = ensemble_models[best_model_name]
else:
    final_model = best_rf_model

final_model.fit(X_train, y_train)

Best model based on cross-validation F1:
Soft Voting Ensemble


VotingClassifier(estimators=[('random_forest',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=10,
                                                     min_samples_leaf=2,
                                                     n_estimators=50, n_jobs=1,
                                                     random_state=42)),
                             ('extra_trees',
                              ExtraTreesClassifier(class_weight='balanced',
                                                   max_depth=10,
                                                   min_samples_leaf=2,
                                                   n_estimators=50, n_jobs=1,
                                                   random_state=42)),
                             ('gradient_boosting',
                              GradientBoostingClassifier(n_estimators=50,
                                                         random_state=42)),
                             ('logistic_regression',
                              Pipeline(steps=[('scaler', StandardScaler()),
                                              ('model',
                                               LogisticRegression(class_weight='balanced',
                                                                  max_iter=1000,
                                                                  random_state=42))]))],
                 n_jobs=1, voting='soft')

In [ ]:
# ============================================================
# STEP 19: Evaluate Final Model on Holdout Test Set
# ============================================================

y_pred = final_model.predict(X_test)

if hasattr(final_model, "predict_proba"):
    y_proba = final_model.predict_proba(X_test)[:, 1]
else:
    y_proba = None

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

if y_proba is not None:
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))
    print("PR-AUC:", average_precision_score(y_test, y_proba))

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.88      0.90     11900
           1       0.37      0.48      0.42      1757

    accuracy                           0.83     13657
   macro avg       0.65      0.68      0.66     13657
weighted avg       0.85      0.83      0.84     13657

Confusion Matrix:
[[10481  1419]
 [  906   851]]
Accuracy: 0.8297576334480486
Precision: 0.3748898678414097
Recall: 0.48434832100170744
F1: 0.42264713185994535
ROC-AUC: 0.7803998890392811
PR-AUC: 0.3530269584234888


In [ ]:
# ============================================================
# STEP 20: Save Final Best Model
# ============================================================

final_model_path = os.path.join(OUTPUT_DIR, "final_best_model.pkl")
joblib.dump(final_model, final_model_path)

print("Saved final best model to:")
print(final_model_path)

Saved final best model to:
/content/model_results/final_best_model.pkl


In [ ]:
# ============================================================
# STEP 22: Save Final Results Summary
# ============================================================

final_summary = {
    "best_model": best_model_name,
    "test_accuracy": accuracy_score(y_test, y_pred),
    "test_precision": precision_score(y_test, y_pred),
    "test_recall": recall_score(y_test, y_pred),
    "test_f1": f1_score(y_test, y_pred)
}

if y_proba is not None:
    final_summary["test_roc_auc"] = roc_auc_score(y_test, y_proba)
    final_summary["test_pr_auc"] = average_precision_score(y_test, y_proba)

final_summary_df = pd.DataFrame([final_summary])

summary_path = os.path.join(OUTPUT_DIR, "final_best_model_summary.csv")
final_summary_df.to_csv(summary_path, index=False)

final_summary_df

,best_model,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,Soft Voting Ensemble,0.829758,0.37489,0.484348,0.422647,0.7804,0.353027


## Model Selection Explanation

The primary metric for this project is F1 score because the task is binary classification with potential class imbalance. In deforestation detection, accuracy alone can be misleading because the model may predict the majority class often and still appear accurate.

F1 score balances precision and recall. Recall is important because missing true deforestation patches would reduce the usefulness of the monitoring system. Precision is also important because too many false alarms would reduce trust in the model.

The notebook tested multiple machine learning algorithms, including baseline models, linear models, tree-based models, boosting models, support vector machines, K-nearest neighbors, Naive Bayes, and a neural network classifier.

Cross-validation was performed using 5-fold cross-validation with `cross_validate(cv=5)`, which follows the certification curriculum approach. This gives a more reliable estimate of model performance than a single train/test split.

The project also tuned Random Forest hyperparameters and compared different tree criteria, including Gini, entropy, and log loss. Ensemble models were built using soft voting and stacking to test whether combining models improves performance.

The final model was selected based on the highest cross-validated F1 score, supported by recall, ROC-AUC, PR-AUC, and the overfitting gap between training F1 and validation F1.

The final model achieved an accuracy of 82.98%, ROC-AUC of 0.780, and F1 score of 0.423 on the holdout test set. While overall accuracy was high, the dataset exhibited class imbalance, making F1 score, Recall, and ROC-AUC more informative measures of performance. The model successfully identified approximately 48% of all deforested patches while maintaining an ROC-AUC substantially above random performance. The results indicate that NDVI-derived features contain predictive information for forest-loss detection, but additional spatial and spectral features would likely improve model performance.

Result files location:
https://drive.google.com/drive/folders/1-GWzPjSKbCjFhEOXjpubMa2Jq15b1EDV?usp=sharing